In [10]:
from pathlib import Path

def parse_comet_filename(filename: str, extension: str = ".png") -> str:
    """
    Convert Comet image filenames like:

    Empirical_empirical_vpc_images_lenuzza-2016__end_image_011-6700
    Empirical_empirical_vpc_images_lenuzza-2016__log_rmse_image_000-6700

    into:

    lenuzza_2016_end_011_6700.png
    lenuzza_2016_log_rmse_000_6700.png
    """
    prefix = "Empirical_empirical_vpc_images_"

    if not filename.startswith(prefix):
        raise ValueError(f"Unexpected filename format: {filename}")

    rest = filename[len(prefix):]
    # e.g.
    # lenuzza-2016__end_image_011-6700
    # lenuzza-2016__log_rmse_image_000-6700

    study, tail = rest.split("__", 1)
    study = study.replace("-", "_")

    # split from the right so the pattern is robust
    # tail = "<checkpoint_type>_image_<num1>-<num2>"
    checkpoint_type, _, number_part = tail.rpartition("_image_")

    if not checkpoint_type or not number_part:
        raise ValueError(f"Could not parse checkpoint type / number part: {filename}")

    n1, n2 = number_part.split("-")

    return f"{study}_{checkpoint_type}_{n1}_{n2}{extension}"


def should_skip_file(filename: str) -> bool:
    return (
        "Zone.Identifier" in filename
        or filename.endswith(":Zone.Identifier")
        or "Zone.Identifier" in filename
    )


def get_one_example_rename(folder_path: str, extension: str = ".png") -> tuple[str, str]:
    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(f"Folder does not exist: {folder_path}")
    if not folder.is_dir():
        raise NotADirectoryError(f"Not a folder: {folder_path}")

    for file_path in folder.iterdir():
        if not file_path.is_file():
            continue

        old_name = file_path.name

        if should_skip_file(old_name):
            continue

        new_name = parse_comet_filename(old_name, extension=extension)
        return old_name, new_name

    raise ValueError(f"No valid files found in folder: {folder_path}")


def rename_all_comet_files(folder_path: str, extension: str = ".png", dry_run: bool = True):
    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(f"Folder does not exist: {folder_path}")
    if not folder.is_dir():
        raise NotADirectoryError(f"Not a folder: {folder_path}")

    for file_path in folder.iterdir():
        if not file_path.is_file():
            continue

        old_name = file_path.name

        if should_skip_file(old_name):
            print(f"Skipping metadata file: {old_name}")
            continue

        try:
            new_name = parse_comet_filename(old_name, extension=extension)
        except Exception as e:
            print(f"Skipping unrecognized file: {old_name} ({e})")
            continue

        new_path = file_path.with_name(new_name)

        if dry_run:
            print(f"{old_name}  -->  {new_name}")
        else:
            file_path.rename(new_path)
            print(f"Renamed: {old_name}  -->  {new_name}")

In [9]:
print(parse_comet_filename("Empirical_empirical_vpc_images_lenuzza-2016__end_image_011-6700"))
print(parse_comet_filename("Empirical_empirical_vpc_images_lenuzza-2016__log_rmse_image_000-6700"))

lenuzza_2016_end_011_6700.png
lenuzza_2016_log_rmse_000_6700.png


In [2]:
comet_images_folder = "/home/cesarali/Pharma/pff/reports/images_from_comet/"

In [12]:
rename_all_comet_files(comet_images_folder, dry_run=False)

Renamed: Empirical_empirical_vpc_images_lenuzza-2016__log_rmse_image_001-6700  -->  lenuzza_2016_log_rmse_001_6700.png
Skipping metadata file: Empirical_empirical_vpc_images_lenuzza-2016__log_rmse_image_008-6700:Zone.Identifier
Skipping metadata file: Empirical_empirical_vpc_images_lenuzza-2016__log_rmse_image_014-6700 (1):Zone.Identifier
Renamed: Empirical_empirical_vpc_images_lenuzza-2016__log_rmse_image_006-6700 (1)  -->  lenuzza_2016_log_rmse_006_6700 (1).png
Renamed: Empirical_empirical_vpc_images_lenuzza-2016__log_rmse_image_011-6700  -->  lenuzza_2016_log_rmse_011_6700.png
Renamed: Empirical_empirical_vpc_images_lenuzza-2016__log_rmse_image_004-6700  -->  lenuzza_2016_log_rmse_004_6700.png
Renamed: Empirical_empirical_vpc_images_lenuzza-2016__log_rmse_image_013-6700 (1)  -->  lenuzza_2016_log_rmse_013_6700 (1).png
Renamed: Empirical_empirical_vpc_images_lenuzza-2016__log_rmse_image_008-6700  -->  lenuzza_2016_log_rmse_008_6700.png
Renamed: Empirical_empirical_vpc_images_lenuzza-

In [13]:
from pathlib import Path
comet_images_folder = "/home/cesarali/Pharma/pff/reports/images_from_comet/"

folder = Path(comet_images_folder)

for file_path in sorted(folder.iterdir()):
    if file_path.is_file():
        print(file_path.name)

Indometacin_log_rmse_000_670.png
Theophylline_log_rmse_000_6700.png
lenuzza_2016_log_rmse_000_6700.png
lenuzza_2016_log_rmse_001_6700.png
lenuzza_2016_log_rmse_002_6700.png
lenuzza_2016_log_rmse_003_6700.png
lenuzza_2016_log_rmse_004_6700.png
lenuzza_2016_log_rmse_005_6700.png
lenuzza_2016_log_rmse_006_6700.png
lenuzza_2016_log_rmse_007_6700.png
lenuzza_2016_log_rmse_008_6700.png
lenuzza_2016_log_rmse_009_6700.png
lenuzza_2016_log_rmse_010_6700.png
lenuzza_2016_log_rmse_011_6700.png
lenuzza_2016_log_rmse_012_6700.png
lenuzza_2016_log_rmse_013_6700.png
lenuzza_2016_log_rmse_014_6700.png
lenuzza_2016_log_rmse_015_6700.png
lenuzza_2016_log_rmse_016_6700.png
lenuzza_2016_log_rmse_017_6700.png
